<a href="https://colab.research.google.com/github/awinarko-hue/Project_hue/blob/main/Legal/Ekstrak/Exp01_R1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import os, re, glob, json, time, pickle
import numpy as np
import pandas as pd
import networkx as nx
pd.set_option('display.max_colwidth', 120)
print("Setup selesai.")

Setup selesai.


In [14]:
MOUNT_DRIVE = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR   = '/content/drive/MyDrive/Project 7/'   # sesuaikan
else:
    BASE_DIR   = '/content/drive/MyDrive/Project 7/Experiment'

# Menghilangkan '/' di awal 'output_parser' agar os.path.join menggabungkannya dengan benar
KORPUS_DIR   = os.path.join(BASE_DIR, 'output_parser')
IFACT_DIR = os.path.join(BASE_DIR, 'artifacts')   # indeks FAISS, graf, embedding
os.makedirs(KORPUS_DIR, exist_ok=True)
os.makedirs(IFACT_DIR, exist_ok=True) # Changed ARTIFACT_DIR to IFACT_DIR

# Model embedding multilingual
#EMBED_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'

print("Korpus dicari di:", KORPUS_DIR)
print("File terdeteksi :", sorted(os.path.basename(f) for f in glob.glob(os.path.join(KORPUS_DIR, 'korpus*.csv'))))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Korpus dicari di: /content/drive/MyDrive/Project 7/output_parser
File terdeteksi : ['korpusPerdirjen_3_2024.csv', 'korpusPerdirjen_7_2021.csv']


In [15]:
SCHEMA = ['doc_id','nama_dokumen','jenis','tahun','nomor','bab','nama_bab','bagian',
          'paragraf','pasal','ayat','huruf','angka','tipe','isi','referensi','kata_kunci','entitas']

def load_korpus(korpus_dir: str) -> pd.DataFrame:
    frames, laporan = [], []
    for path in sorted(glob.glob(os.path.join(korpus_dir, 'korpus*.csv'))):
        df = pd.read_csv(path)
        hilang = [c for c in SCHEMA if c not in df.columns]
        if hilang:
            print(f"[LEWATI] {os.path.basename(path)} — kolom hilang: {hilang}")
            continue
        df = df[SCHEMA].copy()
        df['sumber_file'] = os.path.basename(path)
        frames.append(df)
        laporan.append((os.path.basename(path), len(df), df.doc_id.iloc[0]))
    korpus = pd.concat(frames, ignore_index=True)
    print(pd.DataFrame(laporan, columns=['file','baris','doc_id']).to_string(index=False))
    return korpus

def bersihkan(korpus: pd.DataFrame) -> pd.DataFrame:
    df = korpus.copy()
    # 1. Normalisasi teks
    df['isi'] = (df['isi'].astype(str)
                   .str.replace(r'\s+', ' ', regex=True)
                   .str.strip())
    df = df[df['isi'].str.len() > 0].copy()
    # 2. Standarisasi metadata
    df['jenis'] = df['jenis'].astype(str).str.strip()
    df['tahun'] = pd.to_numeric(df['tahun'], errors='coerce').astype('Int64')
    for c in ['pasal','ayat','angka']:
        df[c] = (df[c].astype('string').str.strip()
                       .str.replace(r'\.0$', '', regex=True))   # '3.0' → '3'
    df['huruf'] = df['huruf'].astype('string').str.strip().str.lower()
    # 3. Buang duplikat persis
    n0 = len(df)
    df = df.drop_duplicates(subset=['doc_id','pasal','ayat','huruf','angka','isi'])
    if n0 - len(df): print(f"Duplikat dibuang: {n0-len(df)}")
    # 4. chunk_id unik & hirarkis, mis. UU_6_2023__P154A__a1__h-g__n4
    def buat_id(r):
        parts = [str(r['doc_id']), f"P{r['pasal']}" if pd.notna(r['pasal']) else "P-"]
        if pd.notna(r['ayat']):  parts.append(f"a{r['ayat']}")
        if pd.notna(r['huruf']): parts.append(f"h{r['huruf']}")
        if pd.notna(r['angka']): parts.append(f"n{r['angka']}")
        return "__".join(parts)
    df['chunk_id'] = df.apply(buat_id, axis=1)
    # jika masih ada bentrok id (unit sama terpecah beberapa baris), beri sufiks urut
    dup = df.duplicated('chunk_id', keep=False)
    df.loc[dup, 'chunk_id'] = (df.loc[dup, 'chunk_id'] + '__' +
                               df.loc[dup].groupby('chunk_id').cumcount().astype(str))
    df = df.reset_index(drop=True)
    return df

korpus = bersihkan(load_korpus(KORPUS_DIR))
print(f"\nTotal satuan hukum: {len(korpus)} | Dokumen: {korpus.doc_id.nunique()}")
korpus.head(3)

                      file  baris            doc_id
korpusPerdirjen_3_2024.csv    781  Perdirjen_3_2024
korpusPerdirjen_7_2021.csv   1886 Perdirjen_07_2021
Duplikat dibuang: 472

Total satuan hukum: 2195 | Dokumen: 2


,doc_id,nama_dokumen,jenis,tahun,nomor,bab,nama_bab,bagian,paragraf,pasal,ayat,huruf,angka,tipe,isi,referensi,kata_kunci,entitas,sumber_file,chunk_id
0,Perdirjen_3_2024,Perdirjen Nomor 3 Tahun 2024,Perdirjen,2024,3,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,<NA>,Definisi,Dalam Peraturan Direktur Jenderal ini yang dimaksud dengan:,Pasal 1,"direktur, jenderal",NaN,korpusPerdirjen_3_2024.csv,Perdirjen_3_2024__P1
1,Perdirjen_3_2024,Perdirjen Nomor 3 Tahun 2024,Perdirjen,2024,3,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,1,Ketentuan,"Telekomunikasi adalah setiap pemancaran, pengiriman, dan/atau penerimaan dari hasil informasi dalam bentuk tanda-tan...",Pasal 1 angka 1,"tanda, sistem, telekomunikasi, setiap, pemancaran",NaN,korpusPerdirjen_3_2024.csv,Perdirjen_3_2024__P1__n1
2,Perdirjen_3_2024,Perdirjen Nomor 3 Tahun 2024,Perdirjen,2024,3,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,2,Ketentuan,Spektrum Frekuensi Radio adalah gelombang elektromagnetik dengan frekuensi lebih kecil dari 3000 GHz yang merambat d...,Pasal 1 angka 2,"frekuensi, penyelenggaraan, spektrum, radio, gelombang",NaN,korpusPerdirjen_3_2024.csv,Perdirjen_3_2024__P1__n2


In [16]:
POLA_INTERNAL  = re.compile(r'Pasal\s+(\d+[A-Z]*)(?:\s+ayat\s*\((\d+)\))?', re.I)
POLA_EKSTERNAL = re.compile(
    r'(Undang-Undang'
    r'|Peraturan +Pemerintah(?: +Pengganti +Undang-Undang)?'
    r'|Peraturan +Presiden'
    r'|(?:Peraturan +Menteri|Peraturan +Direktur +Jenderal|Keputusan +Menteri)'
    r'(?: +(?!Nomor[ ,])[A-Za-z&,]+)*)'
    r' +Nomor +([0-9]+) +Tahun +([12][0-9][0-9][0-9])', re.I)

JENIS_MAP = {'undang-undang':'UU', 'peraturan pemerintah':'PP',
             'peraturan pemerintah pengganti undang-undang':'Perppu',
             'peraturan presiden':'Perpres', 'peraturan menteri':'Permen'}

def ekstrak_rujukan(df: pd.DataFrame):
    internal, eksternal = [], []
    doc_lookup = {}   # (jenis, nomor, tahun) -> doc_id
    for _, r in df.drop_duplicates('doc_id').iterrows():
        doc_lookup[(str(r['jenis']).lower(), str(r['nomor']), str(r['tahun']))] = r['doc_id']
    pasal_ada = set(zip(df['doc_id'], df['pasal'].astype(str)))
    for _, r in df.iterrows():
        teks = r['isi']
        for m in POLA_INTERNAL.finditer(teks):
            p_tuju = m.group(1)
            if str(p_tuju) != str(r['pasal']):     # abaikan rujukan ke diri sendiri
                internal.append({'chunk_id': r['chunk_id'], 'doc_id': r['doc_id'],
                                 'pasal_asal': r['pasal'], 'pasal_tuju': p_tuju,
                                 'ayat_tuju': m.group(2),
                                 'valid': (r['doc_id'], p_tuju) in pasal_ada})
        for m in POLA_EKSTERNAL.finditer(teks):
            jenis_key = re.sub(r'\s+', ' ', m.group(1).lower())
            jenis = JENIS_MAP.get(jenis_key, m.group(1))
            key = (jenis.lower(), m.group(2), m.group(3))
            tuju = None
            for (j, n, t), did in doc_lookup.items():
                if n == m.group(2) and t == m.group(3):
                    tuju = did; break
            eksternal.append({'chunk_id': r['chunk_id'], 'doc_id': r['doc_id'],
                              'jenis_tuju': jenis, 'nomor': m.group(2), 'tahun': m.group(3),
                              'doc_tuju': tuju, 'dalam_korpus': tuju is not None})
    return pd.DataFrame(internal), pd.DataFrame(eksternal)

ref_internal, ref_eksternal = ekstrak_rujukan(korpus)

print("=== Rujukan internal antar-pasal ===")
print(f"Total: {len(ref_internal)} | valid (pasal tujuan ada): {ref_internal['valid'].sum()}"
      f" | tak ditemukan: {(~ref_internal['valid']).sum()}")
print("\n=== Rujukan antar-dokumen ===")
print(f"Total sebutan: {len(ref_eksternal)} | menuju dokumen dalam korpus: {ref_eksternal['dalam_korpus'].sum()}")
print("\nRegulasi luar korpus yang paling sering dirujuk (kandidat perluasan korpus):")
print(ref_eksternal[~ref_eksternal.dalam_korpus]
      .groupby(['jenis_tuju','nomor','tahun']).size().sort_values(ascending=False).head(10))

=== Rujukan internal antar-pasal ===
Total: 451 | valid (pasal tujuan ada): 372 | tak ditemukan: 79

=== Rujukan antar-dokumen ===
Total sebutan: 84 | menuju dokumen dalam korpus: 6

Regulasi luar korpus yang paling sering dirujuk (kandidat perluasan korpus):
jenis_tuju                                                                 nomor  tahun
UU                                                                         8      1981     23
                                                                           36     1999     22
PP                                                                         5      2021     19
PERATURAN DIREKTUR JENDERAL SUMBER DAYA DAN PERANGKAT POS DAN INFORMATIKA  07     2021      6
PP                                                                         43     2012      2
Peraturan Menteri Komunikasi dan Informatika                               16     2018      2
PP                                                                         46     2021  

In [18]:
korpus['nomor'] = pd.to_numeric(korpus['nomor'], errors='coerce').astype('Int64') # menunjukkan bahwa kolom nomor dalam DataFrame korpus memiliki data yang tidak dapat dikonversi ke tipe integer saat menyimpan ke format Parquet.

korpus.to_parquet(os.path.join(IFACT_DIR, 'korpus_bersih.parquet'))
ref_internal.to_csv(os.path.join(IFACT_DIR, 'ref_internal.csv'), index=False)
ref_eksternal.to_csv(os.path.join(IFACT_DIR, 'ref_eksternal.csv'), index=False)
print("Artefak Tahap 1 tersimpan di", IFACT_DIR)

Artefak Tahap 1 tersimpan di /content/drive/MyDrive/Project 7/artifacts


# Knowledge Graph

In [19]:
KG = nx.MultiDiGraph()

def id_bab(doc, bab):     return f"{doc}::{bab}"
def id_pasal(doc, pasal): return f"{doc}::P{pasal}"

# 1) Hirarki struktural (idempoten: key tetap mencegah sisi paralel)
for _, r in korpus.iterrows():
    d = r['doc_id']
    b = id_bab(d, r['bab'])
    p = id_pasal(d, r['pasal'])
    tahun_value = int(r['tahun']) if pd.notna(r['tahun']) else None

    KG.add_node(d, tipe='dokumen', label=r['nama_dokumen'], jenis=r['jenis'], tahun=tahun_value)
    KG.add_node(b, tipe='bab', label=str(r['nama_bab']))
    KG.add_node(p, tipe='pasal', label='Pasal ' + str(r['pasal']))
    KG.add_node(r['chunk_id'], tipe='satuan', norma=r['tipe'], pasal=str(r['pasal']))

    KG.add_edge(d, b, key='memuat', rel='memuat')
    KG.add_edge(b, p, key='memuat', rel='memuat')
    KG.add_edge(p, r['chunk_id'], key='memuat', rel='memuat')

# 2) Rujukan internal: satu sisi per pasangan pasal
pasangan = (ref_internal[ref_internal.valid]
            .drop_duplicates(['doc_id', 'pasal_asal', 'pasal_tuju']))
for _, r in pasangan.iterrows():
    KG.add_edge(id_pasal(r['doc_id'], r['pasal_asal']),
                id_pasal(r['doc_id'], r['pasal_tuju']),
                key='merujuk', rel='merujuk')

# 3) Rujukan antar-dokumen
sub = ref_eksternal[ref_eksternal.dalam_korpus].drop_duplicates(['doc_id', 'doc_tuju'])
for _, r in sub.iterrows():
    if r['doc_id'] != r['doc_tuju']:
        KG.add_edge(r['doc_id'], r['doc_tuju'], key='merujuk_dokumen', rel='merujuk_dokumen')

# 4) Entitas (key mencegah sisi ganda bila entitas sama muncul dua kali)
for _, r in korpus[korpus['entitas'].notna()].iterrows():
    for ent in str(r['entitas']).split(','):
        ent = ent.strip()
        if ent:
            eid = 'ENT::' + ent
            KG.add_node(eid, tipe='entitas', label=ent)
            KG.add_edge(r['chunk_id'], eid, key='menyebut', rel='menyebut')

print(f"Simpul: {KG.number_of_nodes():,} | Sisi: {KG.number_of_edges():,}")
print(pd.Series([d['tipe'] for _, d in KG.nodes(data=True)]).value_counts().to_string())
print()
print(pd.Series([d['rel'] for _, _, d in KG.edges(data=True)]).value_counts().to_string())

with open(os.path.join(IFACT_DIR, 'knowledge_graph.gpickle'), 'wb') as f:
    pickle.dump(KG, f)

Simpul: 2,384 | Sisi: 2,650
satuan     2195
pasal       152
bab          22
entitas      13
dokumen       2

memuat             2369
merujuk             146
menyebut            134
merujuk_dokumen       1


In [20]:
!pip install networkx pyvis

# Statistik graf untuk laporan tesis + visualisasi sampel
deg = pd.Series({n: KG.degree(n) for n, d in KG.nodes(data=True) if d['tipe'] == 'pasal'})
print("Pasal paling terhubung (hub regulasi):")
for n, v in deg.sort_values(ascending=False).head(8).items():
    print(f"  {n}  (derajat {v})")

# Visualisasi interaktif subgraf sekitar satu pasal (buka file HTML hasilnya)
from pyvis.network import Network
def gambar_subgraf(node_pusat: str, radius: int = 1, out='subgraf.html'):
    simpul = {node_pusat}
    for _ in range(radius):
        tetangga = set()
        for s in simpul:
            tetangga |= set(KG.successors(s)) | set(KG.predecessors(s))
        simpul |= tetangga
    sub = KG.subgraph(simpul)
    warna = {'dokumen':'#d62728','bab':'#ff7f0e','pasal':'#1f77b4','satuan':'#2ca02c','entitas':'#9467bd'}
    net = Network(height='600px', directed=True, notebook=False, cdn_resources='remote')
    for n, d in sub.nodes(data=True):
        net.add_node(n, label=d.get('label', n)[:40], color=warna[d['tipe']], title=d['tipe'])
    for u, v, d in sub.edges(data=True):
        net.add_edge(u, v, label=d['rel'])
    net.save_graph(out); print("Tersimpan:", out)

gambar_subgraf(deg.idxmax(), radius=1)

Pasal paling terhubung (hub regulasi):
  Perdirjen_07_2021::P75  (derajat 712)
  Perdirjen_3_2024::P5  (derajat 53)
  Perdirjen_3_2024::P1  (derajat 47)
  Perdirjen_3_2024::P56  (derajat 41)
  Perdirjen_07_2021::P53  (derajat 41)
  Perdirjen_07_2021::P1  (derajat 40)
  Perdirjen_3_2024::P61  (derajat 32)
  Perdirjen_07_2021::P39  (derajat 31)
Tersimpan: subgraf.html


In [21]:
G_rujuk = nx.DiGraph([(u, v) for u, v, d in KG.edges(data=True) if d['rel'] == 'merujuk'])
masuk  = pd.Series(dict(G_rujuk.in_degree())).sort_values(ascending=False)
keluar = pd.Series(dict(G_rujuk.out_degree())).sort_values(ascending=False)

print('Pasal paling banyak dirujuk:')
for n, v in masuk.head(8).items():
    print('  ', n, '(dirujuk', v, 'kali)')
print('Pasal paling banyak merujuk:')
for n, v in keluar.head(8).items():
    print('  ', n, '(merujuk', v, 'kali)')

gambar_subgraf(masuk.index[0], radius=1)

Pasal paling banyak dirujuk:
   Perdirjen_3_2024::P5 (dirujuk 48 kali)
   Perdirjen_07_2021::P16 (dirujuk 7 kali)
   Perdirjen_3_2024::P19 (dirujuk 6 kali)
   Perdirjen_07_2021::P5 (dirujuk 5 kali)
   Perdirjen_3_2024::P18 (dirujuk 3 kali)
   Perdirjen_3_2024::P2 (dirujuk 3 kali)
   Perdirjen_07_2021::P2 (dirujuk 3 kali)
   Perdirjen_07_2021::P15 (dirujuk 3 kali)
Pasal paling banyak merujuk:
   Perdirjen_07_2021::P75 (merujuk 27 kali)
   Perdirjen_3_2024::P29 (merujuk 3 kali)
   Perdirjen_3_2024::P30 (merujuk 3 kali)
   Perdirjen_3_2024::P9 (merujuk 2 kali)
   Perdirjen_3_2024::P22 (merujuk 2 kali)
   Perdirjen_3_2024::P21 (merujuk 2 kali)
   Perdirjen_3_2024::P18 (merujuk 2 kali)
   Perdirjen_3_2024::P14 (merujuk 2 kali)
Tersimpan: subgraf.html


### Visualisasi Subgraf untuk PP_28_2025

Kode di bawah ini akan membuat visualisasi subgraf yang berpusat pada **Pasal 189 dari PP 28/2025** (pasal paling populer di PP tersebut) agar Anda dapat melihat struktur jaringan pada dokumen PP 28/2025.

In [22]:
# Membuat subgraf yang berpusat pada pasal terpopuler dari PP_28_2025
pusat_pp = 'PP_28_2025::P189'
gambar_subgraf(pusat_pp, radius=1, out='subgraf_pp_28_2025.html')

NetworkXError: The node PP_28_2025::P189 is not in the digraph.

### Visualisasi Hubungan Antar Dokumen (Gabungan)

Kode di bawah ini memfilter simpul bertipe `dokumen` dan `pasal` yang memiliki relasi lintas dokumen, sehingga Anda bisa melihat keterkaitan antara **Perdirjen_3_2024** dan **PP_28_2025** secara langsung.

In [23]:
# Membuat subgraf yang berisi simpul dokumen dan pasal-pasal utama dari kedua dokumen
simpul_terpilih = set()

# Masukkan semua simpul tipe 'dokumen'
for n, d in KG.nodes(data=True):
    if d.get('tipe') == 'dokumen':
        simpul_terpilih.add(n)

# Masukkan pasal-pasal yang memiliki derajat keterhubungan tinggi (top 15)
deg_pasal = pd.Series({n: KG.degree(n) for n, d in KG.nodes(data=True) if d['tipe'] == 'pasal'})
top_pasal = deg_pasal.sort_values(ascending=False).head(15).index
simpul_terpilih.update(top_pasal)

# Tambahkan tetangga dari pasal-pasal top tersebut
for p in top_pasal:
    simpul_terpilih.update(KG.successors(p))
    simpul_terpilih.update(KG.predecessors(p))

# Filter graf hanya untuk simpul terpilih
sub_gabungan = KG.subgraph(simpul_terpilih)

# Visualisasikan
warna = {'dokumen':'#d62728', 'bab':'#ff7f0e', 'pasal':'#1f77b4', 'satuan':'#2ca02c', 'entitas':'#9467bd'}
net_gabungan = Network(height='700px', width='100%', directed=True, notebook=False, cdn_resources='remote')

for n, d in sub_gabungan.nodes(data=True):
    net_gabungan.add_node(n, label=d.get('label', n)[:40], color=warna.get(d.get('tipe'), '#97c2fc'), title=d.get('tipe'))

for u, v, d in sub_gabungan.edges(data=True):
    net_gabungan.add_edge(u, v, label=d['rel'])

net_gabungan.save_graph('subgraf_gabungan.html')
print("Tersimpan: subgraf_gabungan.html")

Tersimpan: subgraf_gabungan.html


In [24]:
import networkx as nx
import pandas as pd

# Menggunakan objek KG (Knowledge Graph) yang telah didefinisikan sebelumnya

print("=== RINGKASAN KNOWLEDGE GRAPH ===")
print("Tipe graf       :", type(KG).__name__)
print("Jumlah node     :", KG.number_of_nodes())
print("Jumlah edge     :", KG.number_of_edges())
print("Jumlah komponen terhubung:",
      nx.number_weakly_connected_components(KG)
      if KG.is_directed() else nx.number_connected_components(KG))

# Distribusi tipe node
node_types = pd.Series([
    attrs.get("tipe", "unknown")  # Diubah dari 'type' ke 'tipe' sesuai definisi di cell e02052d6
    for _, attrs in KG.nodes(data=True)
]).value_counts()

print("\nDistribusi tipe node:")
print(node_types)

# Distribusi tipe relasi
edge_types = pd.Series([
    attrs.get("rel", "unknown")   # Diubah dari 'relation'/'label' ke 'rel' sesuai definisi di cell e02052d6
    for _, _, attrs in KG.edges(data=True)
]).value_counts()

print("\nDistribusi tipe relasi:")
print(edge_types)

# Node tanpa edge
isolated = list(nx.isolates(KG))
print("\nJumlah isolated nodes:", len(isolated))

# Duplikasi edge berdasarkan pasangan sumber-tujuan-jenis relasi
edge_records = []
for u, v, key, attrs in KG.edges(keys=True, data=True):
    relation = attrs.get("rel", "unknown")
    edge_records.append((u, v, relation))

edge_df = pd.DataFrame(
    edge_records, columns=["source", "target", "relation"]
)

duplicates = edge_df[
    edge_df.duplicated(
        subset=["source", "target", "relation"],
        keep=False
    )
].sort_values(["source", "target", "relation"])

print("Jumlah edge dalam kelompok duplikat:",
      len(duplicates))
display(duplicates.head(20))

=== RINGKASAN KNOWLEDGE GRAPH ===
Tipe graf       : MultiDiGraph
Jumlah node     : 2384
Jumlah edge     : 2650
Jumlah komponen terhubung: 1

Distribusi tipe node:
satuan     2195
pasal       152
bab          22
entitas      13
dokumen       2
Name: count, dtype: int64

Distribusi tipe relasi:
memuat             2369
merujuk             146
menyebut            134
merujuk_dokumen       1
Name: count, dtype: int64

Jumlah isolated nodes: 0
Jumlah edge dalam kelompok duplikat: 0


,source,target,relation
